# Text-to-SQL post-training: SFT → execution-verified DPO

Runs the full experiment on a single GPU and writes the results table to `results/summary.md`.

**Before you start (Kaggle):** Settings → Accelerator → **GPU T4 x2** (or P100), and turn **Internet on**.
Use **Save Version → Save & Run All (Commit)** so it keeps running after you close the tab; everything in
`/kaggle/working` is kept as the notebook's output.

Every stage is skipped if its output already exists, so re-running continues where it stopped.

In [ ]:
import os, subprocess, sys
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # one GPU keeps the Trainer simple
os.environ["TOKENIZERS_PARALLELISM"] = "false"

REPO_URL = "https://github.com/zimbaan/text2sql-post-training.git"   # change if your repo lives elsewhere
WORK = "/kaggle/working/text2sql-post-training"
MODEL = "Qwen/Qwen2.5-Coder-0.5B-Instruct"
HOLDOUT = 2000

if not os.path.exists(WORK):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, WORK], check=True)
os.chdir(WORK)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
# Kaggle's image ships an old torchao, which makes PEFT refuse to load; this project doesn't use it.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)

In [ ]:
import torch, transformers, trl, peft
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("torch", torch.__version__, "| transformers", transformers.__version__, "| trl", trl.__version__, "| peft", peft.__version__)
!python -m pytest -q tests

In [ ]:
import shlex, time

def stage(done_file, cmd):
    """Run a pipeline step unless its output already exists."""
    if os.path.exists(done_file):
        print("skip:", done_file, "exists")
        return
    print(">>>", cmd, flush=True)
    start = time.time()
    subprocess.run([sys.executable, "-m", *shlex.split(cmd)], check=True)
    print(f"done in {(time.time() - start) / 60:.1f} min")

## 1. Baseline: the untrained model

In [ ]:
stage("results/base.json", f"t2s.evaluate --model {MODEL} --tag base")

## 2. SFT with LoRA (2,000 training questions held out for DPO)

In [ ]:
stage("runs/sft/train_meta.json", f"t2s.train_sft --model {MODEL} --holdout {HOLDOUT} --output runs/sft")
stage("results/sft.json", f"t2s.evaluate --model {MODEL} --adapter runs/sft --tag sft")

## 3. Ablation: SFT on 25% of the data

In [ ]:
stage("runs/sft-25/train_meta.json", f"t2s.train_sft --model {MODEL} --holdout {HOLDOUT} --fraction 0.25 --output runs/sft-25")
stage("results/sft-25.json", f"t2s.evaluate --model {MODEL} --adapter runs/sft-25 --tag sft-25")

## 4. Mine preference pairs and train DPO
The SFT model samples 8 queries per held-out question; each is executed and checked against the gold result.

In [ ]:
stage("data/pairs.jsonl", f"t2s.make_pairs --model {MODEL} --adapter runs/sft --holdout {HOLDOUT} --output data/pairs.jsonl")
print(open("data/pairs.stats.json").read())
stage("runs/dpo/train_meta.json", f"t2s.train_dpo --model {MODEL} --adapter runs/sft --pairs data/pairs.jsonl --output runs/dpo")
stage("results/dpo.json", f"t2s.evaluate --model {MODEL} --adapter runs/dpo --tag dpo")

## 5. Control: plain SFT on the same extra questions
Does DPO on the 2,000 held-out questions beat simply adding their gold SQL to SFT?

In [ ]:
stage("runs/sft-all/train_meta.json", f"t2s.train_sft --model {MODEL} --holdout 0 --output runs/sft-all")
stage("results/sft-all.json", f"t2s.evaluate --model {MODEL} --adapter runs/sft-all --tag sft-all")

## 6. Results

In [ ]:
subprocess.run([sys.executable, "-m", "t2s.report", "--runs", "base", "sft-25", "sft", "sft-all", "dpo"], check=True)
from IPython.display import Image, Markdown, display
display(Markdown(open("results/summary.md").read()))
display(Image("results/accuracy.png"))

In [ ]:
# Keep results, pair stats and the small LoRA adapters as notebook output.
import shutil
out = "/kaggle/working/outputs"
shutil.copytree("results", f"{out}/results", dirs_exist_ok=True)
shutil.copy("data/pairs.stats.json", f"{out}/results/pairs.stats.json")
for run in ["sft", "sft-25", "dpo", "sft-all"]:
    if os.path.exists(f"runs/{run}"):
        shutil.copytree(f"runs/{run}", f"{out}/adapters/{run}", dirs_exist_ok=True,
                        ignore=shutil.ignore_patterns("checkpoint-*", "training_args.bin"))
print(subprocess.run(["du", "-sh", out], capture_output=True, text=True).stdout)